# 누락된 데이터 처리

많은 튜토리얼에서 발견되는 데이터와 실제 데이터의 차이점은 실제 데이터가 깨끗하거나 동질적인 경우가 거의 없다는 것입니다.
특히 많은 흥미로운 데이터 세트에는 어느 정도의 데이터가 누락되어 있습니다.
문제를 더욱 복잡하게 만드는 것은 서로 다른 데이터 소스가 서로 다른 방식으로 누락된 데이터를 나타낼 수 있다는 것입니다.

이번 장에서는 누락된 데이터에 대한 몇 가지 일반적인 고려 사항을 논의하고, Pandas가 이를 표현하기 위해 어떻게 선택하는지 살펴보고, 파이썬(Python)에서 누락된 데이터를 처리하기 위해 내장된 Pandas 도구를 살펴보겠습니다.
여기와 책 전체에서 저는 누락된 데이터를 일반적으로 *null*, *NaN* 또는 *NA* 값으로 지칭할 것입니다.

## 누락된 데이터 규칙의 절충안

테이블이나 `DataFrame`에서 누락된 데이터의 존재를 추적하기 위한 다양한 접근 방식이 개발되었습니다.
일반적으로 누락된 값을 전체적으로 나타내는 *마스크*를 사용하거나 누락된 항목을 나타내는 *감시 값*을 선택하는 두 가지 전략 중 하나를 중심으로 진행됩니다.

마스킹 접근 방식에서 마스크는 완전히 별도의 부울 배열일 수도 있고, 값의 null 상태를 로컬로 나타내기 위해 데이터 표현에서 1비트를 할당하는 작업이 포함될 수도 있습니다.

센티넬 접근 방식에서 센티넬 값은 –9999 또는 일부 희귀 비트 패턴으로 누락된 정수 값을 나타내는 것과 같은 일부 데이터별 규칙일 수도 있고, IEEE 부동 소수점 사양의 일부인 특수 값인 'NaN'(숫자가 아님)으로 누락된 부동 소수점 값을 나타내는 것과 같은 보다 전역적인 규칙일 수도 있습니다.

이러한 접근 방식 중 어느 것도 장단점이 없습니다. 별도의 마스크 배열을 사용하려면 추가 부울 배열을 할당해야 하며, 이는 저장 및 계산 모두에 오버헤드를 추가합니다. Sentinel 값은 표시할 수 있는 유효한 값의 범위를 줄이고 'NaN'과 같은 일반적인 특수 값을 모든 데이터 유형에 사용할 수 없기 때문에 CPU 및 GPU 연산에 추가(종종 최적화되지 않은) 논리가 필요할 수 있습니다.

보편적으로 최적의 선택이 존재하지 않는 대부분의 경우와 마찬가지로 언어와 시스템마다 서로 다른 규칙을 사용합니다.
예를 들어 R 언어는 누락된 데이터를 나타내는 감시 값으로 각 데이터 유형 내의 예약된 비트 패턴을 사용하는 반면, SciDB 시스템은 NA 상태를 나타내기 위해 모든 셀에 첨부된 추가 바이트를 사용합니다.

## Pandas에서 데이터가 누락되었습니다.

Pandas가 누락된 값을 처리하는 방식은 부동 소수점이 아닌 데이터 유형에 대한 NA 값의 내장 개념이 없는 NumPy 패키지에 대한 의존으로 인해 제한됩니다.

아마도 Pandas는 null을 나타내기 위해 각 개별 데이터 유형에 대한 비트 패턴을 지정하는 R의 리드를 따를 수도 있었지만 이 접근 방식은 다소 다루기 힘든 것으로 나타났습니다.
R에는 4개의 주요 데이터 유형만 있지만 NumPy는 이보다 *훨씬* 더 많은 것을 지원합니다. 예를 들어 R에는 단일 정수 유형이 있는 반면 NumPy는 사용 가능한 비트 폭, 부호 있는 인코딩 및 인코딩의 엔디안을 고려하면 14개의 기본 정수 유형을 지원합니다.
사용 가능한 모든 NumPy 유형에서 특정 비트 패턴을 예약하면 다양한 유형에 대한 특수한 작업에서 다루기 힘든 오버헤드가 발생하고 NumPy 패키지의 새로운 포크가 필요할 수도 있습니다. 또한 더 작은 데이터 유형(예: 8비트 정수)의 경우 마스크로 사용하기 위해 비트를 희생하면 표현할 수 있는 값의 범위가 크게 줄어듭니다.

이러한 제약 조건과 장단점으로 인해 Pandas에는 null 값을 저장하고 조작하는 두 가지 "모드"가 있습니다.

- 기본 모드는 데이터 유형에 따라 센티널 값 'NaN' 또는 'None'을 사용하는 센티널 기반 누락 데이터 체계를 사용하는 것입니다.
- 또는 Pandas가 제공하는 nullable 데이터 유형(dtypes)을 사용하도록 선택할 수 있습니다(이 장의 뒷부분에서 설명). 그러면 누락된 항목을 추적하기 위해 동반 마스크 배열이 생성됩니다. 이러한 누락된 항목은 특수 `pd.NA` 값으로 사용자에게 표시됩니다.

두 경우 모두 Pandas API에서 제공하는 데이터 작업 및 조작은 누락된 항목을 예측 가능한 방식으로 처리하고 전파합니다. 그러나 이러한 선택이 *왜* 이루어졌는지에 대한 직관을 개발하기 위해 'None', 'NaN' 및 'NA'에 내재된 장단점을 빠르게 살펴보겠습니다. 평소와 같이 NumPy와 Pandas를 가져오는 것부터 시작하겠습니다.

In [1]:
import numpy as np
import pandas as pd

### Sentinel 값 없음

일부 데이터 유형의 경우 Pandas는 'None'을 센티널 값으로 사용합니다. `None`은 파이썬(Python) 객체입니다. 즉, `None`을 포함하는 모든 배열은 `dtype=object`를 가져야 합니다. 즉, 파이썬(Python) 객체의 시퀀스여야 합니다.

예를 들어 NumPy 배열에 'None'을 전달하면 어떤 일이 발생하는지 관찰해 보세요.

In [2]:
vals1 = np.array([1, None, 2, 3])
vals1

array([1, None, 2, 3], dtype=object)

이 `dtype=object`는 NumPy가 배열의 내용에 대해 추론할 수 있는 가장 일반적인 유형 표현이 파이썬(Python) 객체라는 것을 의미합니다.
이런 방식으로 `None`을 사용하는 경우의 단점은 데이터에 대한 작업이 파이썬(Python) 수준에서 수행되며 기본 유형의 배열에서 일반적으로 나타나는 빠른 작업보다 훨씬 더 많은 오버헤드가 발생한다는 것입니다.

In [3]:
%timeit np.arange(1E6, dtype=int).sum()

2.73 ms ± 288 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [4]:
%timeit np.arange(1E6, dtype=object).sum()

92.1 ms ± 3.42 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


또한 파이썬(Python)은 `None`을 사용한 산술 연산을 지원하지 않기 때문에 `sum` 또는 `min`과 같은 집계는 일반적으로 오류로 이어집니다.

In [5]:
vals1.sum()

TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'

이러한 이유로 Pandas는 숫자 배열에서 'None'을 감시자로 사용하지 않습니다.

### NaN: 숫자 데이터 누락

다른 누락된 데이터 센티널인 'NaN'은 다릅니다. 이는 표준 IEEE 부동 소수점 표현을 사용하는 모든 시스템에서 인식되는 특수 부동 소수점 값입니다.

In [6]:
vals2 = np.array([1, np.nan, 3, 4]) 
vals2

array([ 1., nan,  3.,  4.])

NumPy는 이 배열에 대해 기본 부동 소수점 유형을 선택했습니다. 이는 이전의 객체 배열과 달리 이 배열이 컴파일된 코드에 푸시된 빠른 작업을 지원한다는 것을 의미합니다.
'NaN'은 데이터 바이러스와 약간 비슷하다는 점을 기억하세요. NaN은 접촉하는 다른 모든 개체를 감염시킵니다.
연산에 관계없이 `NaN`을 사용한 산술 결과는 또 다른 `NaN`이 됩니다.

In [7]:
1 + np.nan

nan

In [8]:
0 * np.nan

nan

이는 값에 대한 집계가 잘 정의되어 있지만(즉, 오류가 발생하지 않음) 항상 유용한 것은 아니라는 의미입니다.

In [9]:
vals2.sum(), vals2.min(), vals2.max()

(nan, nan, nan)

즉, NumPy는 이러한 누락된 값을 무시하는 ``NaN`` 인식 버전의 집계를 제공합니다.

In [10]:
np.nansum(vals2), np.nanmin(vals2), np.nanmax(vals2)

(8.0, 1.0, 4.0)

`NaN`의 주요 단점은 이것이 특히 부동 소수점 값이라는 것입니다. 정수, 문자열 또는 기타 유형에 상응하는 `NaN` 값은 없습니다.

### NaN 및 Pandas에는 없음

'NaN'과 'None'은 둘 다 고유한 위치를 갖고 있으며 Pandas는 두 가지를 거의 상호 교환적으로 처리하고 적절한 경우 둘 사이를 변환하도록 만들어졌습니다.

In [11]:
pd.Series([1, np.nan, 2, None])

0    1.0
1    NaN
2    2.0
3    NaN
dtype: float64

사용 가능한 센티널 값이 없는 유형의 경우 NA 값이 있으면 Pandas가 자동으로 유형 변환합니다.
예를 들어 정수 배열의 값을 ``np.nan``으로 설정하면 NA를 수용하기 위해 자동으로 부동 소수점 유형으로 업캐스트됩니다.

In [12]:
x = pd.Series(range(2), dtype=int)
x

0    0
1    1
dtype: int64

In [13]:
x[0] = None
x

0    NaN
1    1.0
dtype: float64

정수 배열을 부동 소수점으로 캐스팅하는 것 외에도 Pandas는 자동으로 ``None``을 ``NaN`` 값으로 변환합니다.

이러한 유형의 마법은 R과 같은 도메인별 언어의 NA 값에 대한 보다 통합된 접근 방식에 비해 약간 해킹적으로 느껴질 수 있지만 Pandas 센티널/캐스팅 접근 방식은 실제로 매우 잘 작동하며 제 경험상 문제가 거의 발생하지 않습니다.

다음 표에는 NA 값이 도입될 때 Pandas의 업캐스팅 규칙이 나열되어 있습니다.

|유형클래스 | NA 저장 시 변환 | NA 센티널 값 |
|---------------|----------------|-----------|
| ``떠있는`` | 변화 없음 | ``np.nan`` |
| ``객체`` | 변화 없음 | ``None`` 또는 ``np.nan`` |
| ``정수`` | ``float64``로 캐스트 | ``np.nan`` |
| ``부울`` | ``객체``로 캐스트 | ``None`` 또는 ``np.nan`` |

Pandas에서 문자열 데이터는 항상 ``object`` dtype으로 저장된다는 점을 명심하세요.

## Pandas Nullable Dtypes

Pandas의 초기 버전에서는 Sentinel 값인 'NaN' 및 'None'이 사용 가능한 유일한 누락 데이터 표현이었습니다. 이로 인해 발생한 주요 어려움은 암시적 유형 캐스팅과 관련이 있었습니다. 예를 들어 데이터가 누락된 실제 정수 배열을 표현할 방법이 없었습니다.

이 문제를 해결하기 위해 Pandas는 나중에 이름을 대문자로 표시하여 일반 dtype과 구별되는 *nullable dtypes*를 추가했습니다(예: `pd.Int32` 대 `np.int32`). 이전 버전과의 호환성을 위해 이러한 nullable dtype은 특별히 요청된 경우에만 사용됩니다.

예를 들어 다음은 누락된 데이터에 대한 사용 가능한 마커 3개를 모두 포함하는 목록에서 생성된 누락된 데이터가 있는 정수 '시리즈'입니다.

In [14]:
pd.Series([1, np.nan, 2, None, pd.NA], dtype='Int32')

0       1
1    <NA>
2       2
3    <NA>
4    <NA>
dtype: Int32

이 표현은 이 장의 나머지 부분에서 살펴보는 모든 작업에서 다른 표현과 상호 교환적으로 사용될 수 있습니다.

## Null 값에 대한 작업

앞서 살펴보았듯이 Pandas는 'None', 'NaN' 및 'NA'를 누락 또는 null 값을 표시하기 위해 기본적으로 상호 교환 가능한 것으로 처리합니다.
이러한 규칙을 용이하게 하기 위해 Pandas는 Pandas 데이터 구조에서 null 값을 감지, 제거 및 대체하는 여러 가지 방법을 제공합니다.
그들은:

- ``isnull``: 누락된 값을 나타내는 부울 마스크를 생성합니다.
- ``notnull``: ``isnull``의 반대말
- ``dropna``: 필터링된 데이터 버전을 반환합니다.
- ``fillna``: 누락된 값이 채워지거나 대치된 데이터의 복사본을 반환합니다.

우리는 이러한 루틴에 대한 간략한 탐구와 시연으로 이 장을 마무리할 것입니다.

### Null 값 감지
Pandas 데이터 구조에는 null 데이터를 감지하는 데 'isnull'과 'notnull'이라는 두 가지 유용한 방법이 있습니다.
둘 중 하나는 데이터에 대한 부울 마스크를 반환합니다. 예를 들어:

In [15]:
data = pd.Series([1, np.nan, 'hello', None])

In [16]:
data.isnull()

0    False
1     True
2    False
3     True
dtype: bool

[데이터 인덱싱 및 선택](03.02-Data-Indexing-and-Selection.ipynb)에서 언급한 대로 부울 마스크는 '시리즈' 또는 'DataFrame' 인덱스로 직접 사용할 수 있습니다.

In [17]:
data[data.notnull()]

0        1
2    hello
dtype: object

`isnull()` 및 `notnull()` 메서드는 ``DataFrame`` 개체에 대해 유사한 부울 결과를 생성합니다.

### Null 값 삭제

이러한 마스킹 방법 외에도 'dropna'라는 편의 방법이 있습니다.
(NA 값 제거) 및 `fillna`(NA 값 채우기). '시리즈'의 경우,
결과는 간단합니다.

In [18]:
data.dropna()

0        1
2    hello
dtype: object

``DataFrame``의 경우 더 많은 옵션이 있습니다.
다음 ``DataFrame``을 고려해보세요:

In [19]:
df = pd.DataFrame([[1,      np.nan, 2],
                   [2,      3,      5],
                   [np.nan, 4,      6]])
df

,0,1,2
0,1.0,NaN,2
1,2.0,3.0,5
2,NaN,4.0,6


`DataFrame`에서 단일 값을 삭제할 수 없습니다. 전체 행이나 열만 삭제할 수 있습니다.
애플리케이션에 따라 둘 중 하나를 원할 수 있으므로 `dropna`에는 `DataFrame`에 대한 다양한 옵션이 포함되어 있습니다.

기본적으로 `dropna`는 *모든* null 값이 존재하는 모든 행을 삭제합니다.

In [20]:
df.dropna()

,0,1,2
1,2.0,3.0,5


또는 다른 축을 따라 NA 값을 삭제할 수 있습니다. `axis=1` 또는 `axis='columns'`를 사용하면 null 값을 포함하는 모든 열이 삭제됩니다.

In [21]:
df.dropna(axis='columns')

,2
0,2
1,5
2,6


그러나 이로 인해 좋은 데이터도 삭제됩니다. *모든* NA 값 또는 대부분의 NA 값이 포함된 행이나 열을 삭제하는 데 관심이 있을 수 있습니다.
이는 통과할 수 있는 null 수를 미세하게 제어할 수 있는 `how` 또는 `thresh` 매개변수를 통해 지정할 수 있습니다.

기본값은 `how='any'`입니다. 따라서 null 값이 포함된 행이나 열은 모두 삭제됩니다.
*all* null 값을 포함하는 행/열만 삭제하는 `how='all'`을 지정할 수도 있습니다.

In [22]:
df[3] = np.nan
df

,0,1,2,3
0,1.0,NaN,2,NaN
1,2.0,3.0,5,NaN
2,NaN,4.0,6,NaN


In [23]:
df.dropna(axis='columns', how='all')

,0,1,2
0,1.0,NaN,2
1,2.0,3.0,5
2,NaN,4.0,6


보다 세부적인 제어를 위해 `thresh` 매개변수를 사용하면 유지할 행/열에 대해 null이 아닌 값의 최소 개수를 지정할 수 있습니다.

In [24]:
df.dropna(axis='rows', thresh=3)

,0,1,2,3
1,2.0,3.0,5,NaN


여기서 첫 번째 행과 마지막 행에는 각각 Null이 아닌 값이 두 개만 포함되어 있으므로 삭제되었습니다.

### Null 값 채우기

때로는 NA 값을 삭제하는 대신 유효한 값으로 바꾸고 싶을 때가 있습니다.
이 값은 0과 같은 단일 숫자일 수도 있고 좋은 값에서 일종의 대치 또는 보간일 수도 있습니다.
`isnull` 메서드를 마스크로 사용하여 내부에서 이 작업을 수행할 수 있지만 이는 일반적인 작업이기 때문에 Pandas는 null 값이 대체된 배열의 복사본을 반환하는 `fillna` 메서드를 제공합니다.

다음 '시리즈'를 고려하세요.

In [25]:
data = pd.Series([1, np.nan, 2, None, 3], index=list('abcde'), dtype='Int32')
data

a       1
b    <NA>
c       2
d    <NA>
e       3
dtype: Int32

NA 항목을 0과 같은 단일 값으로 채울 수 있습니다.

In [26]:
data.fillna(0)

a    1
b    0
c    2
d    0
e    3
dtype: Int32

이전 값을 앞으로 전파하기 위해 정방향 채우기를 지정할 수 있습니다.

In [27]:
# forward fill
data.fillna(method='ffill')

a    1
b    1
c    2
d    2
e    3
dtype: Int32

또는 다음 값을 뒤로 전파하기 위해 뒤로 채우기를 지정할 수 있습니다.

In [28]:
# back fill
data.fillna(method='bfill')

a    1
b    2
c    2
d    3
e    3
dtype: Int32

`DataFrame`의 경우 옵션은 비슷하지만 채우기가 수행되어야 하는 `축`을 지정할 수도 있습니다.

In [29]:
df

,0,1,2,3
0,1.0,NaN,2,NaN
1,2.0,3.0,5,NaN
2,NaN,4.0,6,NaN


In [30]:
df.fillna(method='ffill', axis=1)

,0,1,2,3
0,1.0,1.0,2.0,2.0
1,2.0,3.0,5.0,5.0
2,NaN,4.0,6.0,6.0


정방향 채우기 중에 이전 값을 사용할 수 없는 경우 NA 값이 유지됩니다.